# A07 Input Mels And Stage4 Activations

This notebook compares matched `A07` spoof examples and `bonafide` examples from the same fixed speakers.

For each selected speaker, it plots:
- input mel spectrogram for spoof
- input mel spectrogram for bonafide
- one selected `stage4` channel for spoof
- one selected `stage4` channel for bonafide

The notebook uses the same frame normalization as the TCAV pipeline so the visualized inputs match the actual experiment preprocessing.

## Setup

Load plotting, manifest, audio, and model utilities.

In [ ]:
import io
import tarfile
import tempfile
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchaudio
from IPython.display import Audio, Markdown, display

PROJECT_ROOT = Path('/home/SpeakerRec/BioVoice/redimnet/tcav')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from captum_tcav.common import resolve_layers
from captum_tcav.concepts import infer_target_frames, normalize_frames
from captum_tcav.asvspoof5.config import load_config
from captum_tcav.asvspoof5.modeling import load_model

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

## Config

Define the target system, fixed speakers, and how many matched examples to inspect.

In [ ]:
config = load_config()

DATA_ROOT = Path('/home/SpeakerRec/BioVoice')
PLAN_BASE = DATA_ROOT / 'data' / 'datasets' / 'ASVspoof5_tars' / 'ASVspoof5_protocols' / 'train_dev_16_systems_outputs'
TRAIN_MANIFEST = PLAN_BASE / 'train' / 'selected_utterances_plan.csv'
TRAIN_TAR_DIR = DATA_ROOT / 'data' / 'datasets' / 'ASVspoof5_tars' / 'ASVspoof5_audio_train_tars'
TRAIN_TAR_PREFIX = 'flac_T_*.tar'

SYSTEM_ID = 'A07'
TARGET_SPLIT = 'test'
N_SPEAKERS_TO_PLOT = 4
TARGET_FRAMES = infer_target_frames(config.concept_root, config.concept_names)
STAGE4_CHANNEL = 0

FIXED_TRAIN_SPEAKERS = [
    'T_0380', 'T_0411', 'T_0635', 'T_0897', 'T_1864',
    'T_2149', 'T_2284', 'T_2326', 'T_2791', 'T_3455',
    'T_3714', 'T_3850', 'T_3883', 'T_4049', 'T_4126',
    'T_4175', 'T_4618', 'T_4769', 'T_4913', 'T_5053',
]

print('TRAIN_MANIFEST exists =', TRAIN_MANIFEST.exists())
print('TRAIN_TAR_DIR exists =', TRAIN_TAR_DIR.exists())
print('TARGET_FRAMES =', TARGET_FRAMES)
print('STAGE4_CHANNEL =', STAGE4_CHANNEL)

## Load Manifest

Load the prepared train manifest and isolate fixed-speaker A07 spoof and bonafide test rows.

In [ ]:
manifest = pd.read_csv(TRAIN_MANIFEST)

spoof_df = manifest[
    manifest['split'].eq(TARGET_SPLIT)
    & manifest['target_class'].eq(SYSTEM_ID)
    & manifest['speaker_id'].isin(FIXED_TRAIN_SPEAKERS)
].copy()

bonafide_df = manifest[
    manifest['split'].eq(TARGET_SPLIT)
    & manifest['target_class'].eq('bonafide')
    & manifest['speaker_id'].isin(FIXED_TRAIN_SPEAKERS)
].copy()

speaker_counts = (
    spoof_df.groupby('speaker_id').size().rename('spoof_rows').to_frame()
    .join(bonafide_df.groupby('speaker_id').size().rename('bonafide_rows').to_frame(), how='outer')
    .fillna(0).astype(int)
    .reset_index()
)

matched_speakers = speaker_counts[
    (speaker_counts['spoof_rows'] > 0) & (speaker_counts['bonafide_rows'] > 0)
].copy().sort_values('speaker_id')

display(speaker_counts)
display(matched_speakers.head(N_SPEAKERS_TO_PLOT))

## Build Matched Examples

Pick one spoof and one bonafide utterance for each selected speaker.

In [ ]:
selected_speakers = matched_speakers['speaker_id'].head(N_SPEAKERS_TO_PLOT).tolist()

rows = []
for speaker_id in selected_speakers:
    spoof_row = spoof_df[spoof_df['speaker_id'].eq(speaker_id)].sort_values('utt_id').iloc[0]
    bonafide_row = bonafide_df[bonafide_df['speaker_id'].eq(speaker_id)].sort_values('utt_id').iloc[0]
    rows.append(
        {
            'speaker_id': speaker_id,
            'spoof_utt_id': str(spoof_row['utt_id']),
            'bonafide_utt_id': str(bonafide_row['utt_id']),
        }
    )

matched_examples = pd.DataFrame(rows)
display(matched_examples)

## Audio Helpers

Index the tar shards and define helpers to load waveforms and extract normalized mel and raw stage4 tensors.

In [ ]:
def build_tar_index(tar_dir: Path, tar_prefix: str) -> dict[str, tuple[Path, str]]:
    index = {}
    for tar_path in sorted(tar_dir.glob(tar_prefix)):
        with tarfile.open(tar_path, 'r') as handle:
            for member in handle:
                if not member.isfile():
                    continue
                utt_id = Path(member.name).stem
                if utt_id not in index:
                    index[utt_id] = (tar_path, member.name)
    return index


def load_waveform_from_tar(tar_path: Path, member_name: str) -> tuple[torch.Tensor, int]:
    with tarfile.open(tar_path, 'r') as handle:
        member = handle.getmember(member_name)
        extracted = handle.extractfile(member)
        if extracted is None:
            raise RuntimeError(f'Could not extract {member_name} from {tar_path}')
        raw = extracted.read()
    try:
        return torchaudio.load(io.BytesIO(raw))
    except Exception:
        suffix = Path(member_name).suffix or '.flac'
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=True) as tmp:
            tmp.write(raw)
            tmp.flush()
            return torchaudio.load(tmp.name)


def raw_stage4_tensor(tensor: torch.Tensor) -> np.ndarray:
    arr = tensor.detach().cpu().float().numpy()
    while arr.ndim > 3:
        arr = arr[0]
    return arr


def summarize_mel(tensor: torch.Tensor) -> np.ndarray:
    arr = tensor.detach().cpu().float().numpy()
    while arr.ndim > 3:
        arr = arr[0]
    if arr.ndim == 3:
        arr = arr[0]
    return arr


tar_index = build_tar_index(TRAIN_TAR_DIR, TRAIN_TAR_PREFIX)
print('indexed utt_ids =', len(tar_index))

## Load Model

Load the A07 detector and resolve the internal layer used for TCAV.

In [ ]:
run_config = load_config()
model = load_model(run_config, SYSTEM_ID)
resolved_layers = resolve_layers(model, run_config.layers)
stage4_name = resolved_layers[0]
stage4_module = dict(model.named_modules())[stage4_name]

print('resolved layer =', stage4_name)

## Extract Features

Run matched utterances through the model and capture both normalized mel input and raw stage4 activations.

In [ ]:
def extract_mel_and_stage4(utt_id: str) -> tuple[np.ndarray, np.ndarray, torch.Tensor, int]:
    if utt_id not in tar_index:
        raise KeyError(f'utt_id not found: {utt_id}')
    tar_path, member_name = tar_index[utt_id]
    waveform, sample_rate = load_waveform_from_tar(tar_path, member_name)
    if sample_rate != 16000:
        waveform = torchaudio.functional.resample(waveform, sample_rate, 16000)
        sample_rate = 16000
    waveform = waveform[:1, :].float()

    captured = {}

    def hook_fn(_module, _inputs, output):
        captured['stage4'] = output.detach().cpu()

    handle = stage4_module.register_forward_hook(hook_fn)
    try:
        mel_raw = model.spec(waveform)
        mel_norm = normalize_frames(mel_raw, TARGET_FRAMES)
        _ = model(mel_norm.unsqueeze(0))
    finally:
        handle.remove()

    return summarize_mel(mel_norm), raw_stage4_tensor(captured['stage4']), waveform.squeeze(0).detach().cpu(), sample_rate


example_payload = []
for row in matched_examples.itertuples(index=False):
    spoof_mel, spoof_stage4, spoof_waveform, spoof_sr = extract_mel_and_stage4(row.spoof_utt_id)
    bonafide_mel, bonafide_stage4, bonafide_waveform, bonafide_sr = extract_mel_and_stage4(row.bonafide_utt_id)
    example_payload.append(
        {
            'speaker_id': row.speaker_id,
            'spoof_utt_id': row.spoof_utt_id,
            'bonafide_utt_id': row.bonafide_utt_id,
            'spoof_mel': spoof_mel,
            'bonafide_mel': bonafide_mel,
            'spoof_stage4': spoof_stage4,
            'bonafide_stage4': bonafide_stage4,
            'spoof_waveform': spoof_waveform,
            'bonafide_waveform': bonafide_waveform,
            'spoof_sr': spoof_sr,
            'bonafide_sr': bonafide_sr,
        }
    )

print('loaded matched examples =', len(example_payload))
print('stage4 shape example =', example_payload[0]['spoof_stage4'].shape if example_payload else None)

## Plot Comparison

Plot each speaker separately in a vertical layout so the mel and selected stage4 channel are easier to inspect.

In [ ]:
for item in example_payload:
    fig, axes = plt.subplots(4, 1, figsize=(16, 18), constrained_layout=True)

    im0 = axes[0].imshow(item['spoof_mel'], aspect='auto', origin='lower', cmap='magma')
    axes[0].set_title(f"{item['speaker_id']} | spoof mel\n{item['spoof_utt_id']}")
    axes[0].set_xlabel('time')
    axes[0].set_ylabel('mel bin')
    fig.colorbar(im0, ax=axes[0])

    im1 = axes[1].imshow(item['bonafide_mel'], aspect='auto', origin='lower', cmap='magma')
    axes[1].set_title(f"{item['speaker_id']} | bonafide mel\n{item['bonafide_utt_id']}")
    axes[1].set_xlabel('time')
    axes[1].set_ylabel('mel bin')
    fig.colorbar(im1, ax=axes[1])

    im2 = axes[2].imshow(item['spoof_stage4'][STAGE4_CHANNEL], aspect='auto', origin='lower', cmap='viridis')
    axes[2].set_title(f"{item['speaker_id']} | spoof stage4 channel {STAGE4_CHANNEL}")
    axes[2].set_xlabel('time')
    axes[2].set_ylabel('feature bin')
    fig.colorbar(im2, ax=axes[2])

    im3 = axes[3].imshow(item['bonafide_stage4'][STAGE4_CHANNEL], aspect='auto', origin='lower', cmap='viridis')
    axes[3].set_title(f"{item['speaker_id']} | bonafide stage4 channel {STAGE4_CHANNEL}")
    axes[3].set_xlabel('time')
    axes[3].set_ylabel('feature bin')
    fig.colorbar(im3, ax=axes[3])

    plt.show()

## Listen Samples

Listen to the exact matched spoof and bonafide utterances used in the plots.

In [ ]:
for item in example_payload:
    display(Markdown(f"### {item['speaker_id']}"))
    display(Markdown(f"Spoof: `{item['spoof_utt_id']}`"))
    display(Audio(item['spoof_waveform'].numpy(), rate=item['spoof_sr']))
    display(Markdown(f"Bonafide: `{item['bonafide_utt_id']}`"))
    display(Audio(item['bonafide_waveform'].numpy(), rate=item['bonafide_sr']))

## Read Results

- the first two plots show the normalized mel-like input space actually used before the model forward pass.
- the last two plots show one selected raw `stage4` channel rather than a channel mean.
- change `STAGE4_CHANNEL` in the config cell to inspect different internal feature maps.
- this notebook is diagnostic only; it does not compute TCAV scores.